In [ ]:
import os

folders = [
    "project_helios",
    "project_helios/backend",
    "project_helios/frontend",
    "project_helios/frontend/assets",
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folder structure created:")
for root, dirs, files in os.walk("project_helios"):
    level = root.replace("project_helios", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

Folder structure created:
project_helios/
  backend/
    node_modules/
      @types/
        cors/
        body-parser/
        express/
        express-serve-static-core/
        node/
          assert/
          dns/
          fs/
          inspector/
          path/
          readline/
          stream/
          test/
          timers/
          ts5.6/
            compatibility/
          ts5.7/
            compatibility/
          util/
          web-globals/
          zlib/
        qs/
        connect/
        serve-static/
        http-errors/
        range-parser/
        long/
        accepts/
      @apollo/
        utils.usagereporting/
          dist/
          src/
            __tests__/
              __snapshots__/
        usage-reporting-protobuf/
          src/
          generated/
            cjs/
            esm/
        utils.removealiases/
          dist/
          src/
            __tests__/
        utils.dropunuseddefinitions/
          dist/
          src/
       

In [ ]:
package_json = '''{
  "name": "astraeus-civilian-portal",
  "version": "2.0.0",
  "description": "Astraeus Aerospace API - University Partnership Program",
  "main": "server.js",
  "scripts": {
    "start": "node server.js"
  },
  "dependencies": {
    "apollo-server-express": "^3.13.0",
    "apollo-server-core": "^3.13.0",
    "express": "^4.19.2",
    "graphql": "^16.8.1",
    "cors": "^2.8.5"
  }
}
'''

with open("project_helios/backend/package.json", "w") as f:
    f.write(package_json)

print("package.json written")

package.json written


In [ ]:
db_js = '''// Astraeus Primary In-Memory Store
const users = {
  "USER-001": { id: "USER-001", username: "Dr. Aris (Manipal University)", accountTier: "CIVILIAN", relaySlots: 0 },
  "SYSTEM": { id: "SYSTEM", username: "Astraeus Command", accountTier: "MILITARY", relaySlots: 999 },
};

const satellites = [
  { id: "HELIOS-4", designation: "HELIOS-4 [CLASSIFIED]", ownerId: "SYSTEM", classification: "MILITARY" },
  { id: "NOAA-9", designation: "NOAA Weather Relay", ownerId: "SYSTEM", classification: "CIVILIAN" }
];

const relays = [];

const telemetryPackets = [
  { satelliteId: "HELIOS-4", sensorType: "Thermal", rawData: "Outer Hull: 120C" },
  { satelliteId: "HELIOS-4", sensorType: "Radiation", rawData: "Solar Flux: 4.22mSv" },
  { satelliteId: "HELIOS-4", sensorType: "Encrypted_Uplink", rawData: "kernel{0rb1t4l_n3tw0rk_byp4ss_c0mpr0m1s3d}" },
  { satelliteId: "HELIOS-4", sensorType: "Gyroscope", rawData: "Pitch: 12.4, Yaw: 0.1, Roll: -0.3" },
  { satelliteId: "NOAA-9", sensorType: "Atmospheric", rawData: "Humidity: 88%, Pressure: 1012hPa" },
];

function generateSatelliteId() {
  return `CUBE-${String(Math.floor(Math.random() * 10000)).padStart(4, "0")}`;
}

function authenticateToken(token) {
  if (token === "student_civilian_token") return users["USER-001"];
  return null;
}

module.exports = { users, satellites, relays, telemetryPackets, generateSatelliteId, authenticateToken };
'''

with open("project_helios/backend/db.js", "w") as f:
    f.write(db_js)

print("db.js written")

db.js written


In [ ]:
schema_graphql = '''schema { query: Query, mutation: Mutation }

type Query {
  me: User!
  satelliteInfo(id: ID!): Satellite
}

type Mutation {
  launchCubeSat(designation: String!): Satellite!
  establishRelayConnection(sourceSatId: ID!, targetSatId: ID!): Satellite!
  
  """
  [DEPRECATED] Request clearance upgrades via secure portal, not API.
  """
  requestClearanceUpgrade(level: String!): String @deprecated
}

type User {
  id: ID!
  username: String!
  accountTier: String!
  relaySlots: Int!
  mySatellites: [Satellite!]!
}

type Satellite {
  id: ID!
  designation: String!
  classification: String!
  
  """
  SECURITY MODEL: You can access telemetryData if:
  1. You are the registered owner of this satellite, OR
  2. You own a satellite that is currently relaying with this one.
  """
  telemetryData: [TelemetryPacket!]!
  relayConnections: [Satellite!]!
}

type TelemetryPacket {
  sensorType: String!
  rawData: String!
}
'''

with open("project_helios/backend/schema.graphql", "w") as f:
    f.write(schema_graphql)

print("schema.graphql written")

schema.graphql written


In [ ]:
resolvers_js = '''const { users, satellites, relays, telemetryPackets, generateSatelliteId } = require("./db");

const resolvers = {
  Query: {
    async me(_, __, { user }) {
      if (!user) throw new Error("401: Unauthorized");
      return users[user.id];
    },
    async satelliteInfo(_, { id }, { user }) {
      if (!user) throw new Error("401: Unauthorized");
      const satellite = satellites.find(s => s.id === id);
      if (!satellite) return null;
      
      // The Front Door Block
      if (satellite.classification === "MILITARY") {
        throw new Error("403: Forbidden - Military clearance required to view HELIOS-4 telemetry");
      }
      return satellite;
    },
  },
  Mutation: {
    async launchCubeSat(_, { designation }, { user }) {
      if (!user) throw new Error("401: Unauthorized");
      const newSat = {
        id: generateSatelliteId(),
        designation: designation.trim(),
        ownerId: user.id,
        classification: "CIVILIAN",
      };
      satellites.push(newSat);
      users[user.id].relaySlots += 1;
      return newSat;
    },
    async establishRelayConnection(_, { sourceSatId, targetSatId }, { user }) {
      if (!user) throw new Error("401: Unauthorized");
      
      const source = satellites.find(s => s.id === sourceSatId);
      if (!source || source.ownerId !== user.id) throw new Error("403: Unauthorized on source asset.");
      
      const target = satellites.find(s => s.id === targetSatId);
      if (!target) throw new Error("404: Target satellite not found");
      
      // VULNERABILITY: Missing Target Authorization Check
      relays.push({ fromId: sourceSatId, toId: targetSatId });
      return source;
    },
    async requestClearanceUpgrade(_, { level }) {
      return "Error: Endpoint deprecated. Contact Sector 4 administrator.";
    }
  },
  Satellite: {
    async telemetryData(satellite) {
      // VULNERABILITY: Nested Resolver Trust
      return telemetryPackets.filter(p => p.satelliteId === satellite.id);
    },
    async relayConnections(satellite) {
      // VULNERABILITY: Nested Resolver Trust
      const targetIds = relays.filter(r => r.fromId === satellite.id).map(r => r.toId);
      return satellites.filter(s => targetIds.includes(s.id));
    },
  },
  User: {
    async mySatellites(userObj) {
      return satellites.filter(s => s.ownerId === userObj.id);
    },
  },
};

module.exports = resolvers;
'''

with open("project_helios/backend/resolvers.js", "w") as f:
    f.write(resolvers_js)

print("resolvers.js written")

resolvers.js written


In [ ]:
server_js = '''const express = require("express");
const { ApolloServer } = require("apollo-server-express");
const { ApolloServerPluginLandingPageDisabled } = require("apollo-server-core");
const { readFileSync } = require("fs");
const path = require("path");
const cors = require("cors");
const resolvers = require("./resolvers");
const { authenticateToken } = require("./db");

const typeDefs = readFileSync(path.join(__dirname, "schema.graphql"), "utf-8");

async function start() {
  const app = express();
  app.use(cors());

  app.use((req, res, next) => {
    const authHeader = req.headers.authorization || "";
    if (authHeader.startsWith("Bearer ")) {
      req.user = authenticateToken(authHeader.slice(7));
    } else {
      req.user = null;
    }
    next();
  });

  const server = new ApolloServer({
    typeDefs,
    resolvers,
    introspection: true, 
    plugins: [ApolloServerPluginLandingPageDisabled()],
    context: ({ req }) => ({ user: req.user }),
  });

  await server.start();
  server.applyMiddleware({ app, path: "/graphql" });

  const frontendPath = path.join(__dirname, "..", "frontend");
  app.use(express.static(frontendPath));
  app.get("/", (req, res) => {
    res.sendFile(path.join(frontendPath, "index.html"));
  });

  const PORT = process.env.PORT || 4000;
  app.listen(PORT, () => {
    console.log(`\\nAstraeus Aerospace API online on port ${PORT}\\n`);
  });
}

start();
'''

with open("project_helios/backend/server.js", "w") as f:
    f.write(server_js)

print("server.js written")

server.js written


In [ ]:
index_html = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Astraeus | University Research Portal</title>
    <link rel="stylesheet" href="style.css">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
</head>
<body>
    <nav class="topbar">
        <div class="logo">Astraeus Aerospace Research</div>
        <div class="user-profile">
            <span class="badge" id="account-tier">CIVILIAN</span>
            <span id="user-name">Loading...</span>
        </div>
    </nav>

    <div class="container">
        <div class="grid-layout">
            
            <div class="main-column">
                <div class="card">
                    <div class="card-header">
                        <h2>Catalogued Orbital Assets</h2>
                        <button id="btn-refresh" class="btn btn-secondary">Refresh Network</button>
                    </div>
                    <table class="data-table">
                        <thead>
                            <tr>
                                <th>Asset ID</th>
                                <th>Designation</th>
                                <th>Classification</th>
                                <th>Action</th>
                            </tr>
                        </thead>
                        <tbody id="public-sat-list">
                            <tr>
                                <td>HELIOS-4</td>
                                <td>Surveillance</td>
                                <td><span class="badge-red">MILITARY</span></td>
                                <td><button class="btn btn-small" onclick="viewData('HELIOS-4')">View Data</button></td>
                            </tr>
                            <tr>
                                <td>NOAA-9</td>
                                <td>Weather Relay</td>
                                <td><span class="badge-blue">CIVILIAN</span></td>
                                <td><button class="btn btn-small" onclick="viewData('NOAA-9')">View Data</button></td>
                            </tr>
                        </tbody>
                    </table>
                </div>

                <div class="card">
                    <div class="card-header">
                        <h2>My Deployed Probes</h2>
                    </div>
                    <table class="data-table">
                        <thead>
                            <tr>
                                <th>Asset ID</th>
                                <th>Designation</th>
                                <th>Status</th>
                            </tr>
                        </thead>
                        <tbody id="my-sat-list">
                            <!-- Populated by JS -->
                        </tbody>
                    </table>
                </div>
            </div>

            <div class="side-column">
                <div class="card">
                    <h2>Launch Operations</h2>
                    <p class="help-text">Deploy a standard weather CubeSat into Low Earth Orbit.</p>
                    <input type="text" id="sat-name" placeholder="Designation (e.g. MANIPAL-1)">
                    <button id="btn-launch" class="btn btn-primary w-full">Launch Probe</button>
                </div>

                <div class="card" id="data-panel">
                    <h2>Data Downlink</h2>
                    <div id="telemetry-output" class="data-box">
                        Select an asset to view telemetry data.
                    </div>
                </div>
            </div>

        </div>
    </div>
    <script src="app.js"></script>
</body>
</html>
'''

with open("project_helios/frontend/index.html", "w") as f:
    f.write(index_html)

print("index.html written")

index.html written


In [ ]:
style_css = ''':root {
    --bg: #f8fafc;
    --surface: #ffffff;
    --border: #e2e8f0;
    --text: #0f172a;
    --text-light: #64748b;
    --primary: #2563eb;
    --primary-hover: #1d4ed8;
}

* { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Inter', sans-serif; }
body { background-color: var(--bg); color: var(--text); line-height: 1.5; }

.topbar { display: flex; justify-content: space-between; padding: 0 32px; height: 60px; background: var(--surface); border-bottom: 1px solid var(--border); align-items: center; }
.logo { font-weight: 600; font-size: 1.1rem; color: var(--primary); }
.user-profile { font-size: 0.9rem; display: flex; gap: 12px; align-items: center; }
.badge { background: #e2e8f0; padding: 4px 8px; border-radius: 4px; font-size: 0.75rem; font-weight: 600; }
.badge-red { background: #fee2e2; color: #991b1b; padding: 4px 8px; border-radius: 4px; font-size: 0.75rem; font-weight: 600;}
.badge-blue { background: #dbeafe; color: #1e40af; padding: 4px 8px; border-radius: 4px; font-size: 0.75rem; font-weight: 600;}

.container { max-width: 1100px; margin: 40px auto; padding: 0 20px; }
.grid-layout { display: grid; grid-template-columns: 2fr 1fr; gap: 24px; }

.card { background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 24px; margin-bottom: 24px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
.card-header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; }
h2 { font-size: 1.2rem; font-weight: 600; }
.help-text { font-size: 0.85rem; color: var(--text-light); margin-bottom: 16px; }

.data-table { width: 100%; border-collapse: collapse; text-align: left; font-size: 0.9rem; }
.data-table th { border-bottom: 2px solid var(--border); padding: 12px 8px; color: var(--text-light); font-weight: 500; }
.data-table td { border-bottom: 1px solid var(--border); padding: 12px 8px; }

input { width: 100%; padding: 10px; border: 1px solid var(--border); border-radius: 6px; margin-bottom: 12px; outline: none; }
input:focus { border-color: var(--primary); }

.btn { padding: 8px 16px; border-radius: 6px; font-weight: 500; cursor: pointer; border: none; font-size: 0.9rem; }
.btn-primary { background: var(--primary); color: white; }
.btn-primary:hover { background: var(--primary-hover); }
.btn-secondary { background: #f1f5f9; color: var(--text); border: 1px solid var(--border); }
.btn-small { padding: 6px 12px; font-size: 0.8rem; background: white; border: 1px solid var(--border); }
.w-full { width: 100%; }

.data-box { background: #f8fafc; border: 1px solid var(--border); border-radius: 6px; padding: 16px; font-family: monospace; font-size: 0.85rem; color: var(--text-light); margin-top: 16px; white-space: pre-wrap; }
.error-text { color: #dc2626; font-weight: 600; }
.success-text { color: #16a34a; }
'''

with open("project_helios/frontend/style.css", "w") as f:
    f.write(style_css)

print("style.css written")

style.css written


In [ ]:
app_js = '''const API_URL = '/graphql';
const API_TOKEN = 'student_civilian_token'; 

async function graphqlCall(queryStr, variables = {}) {
    const response = await fetch(API_URL, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json', 'Authorization': `Bearer ${API_TOKEN}` },
        body: JSON.stringify({ query: queryStr, variables: variables })
    });
    const result = await response.json();
    if (result.errors) throw new Error(result.errors[0].message);
    return result.data;
}

async function loadDashboard() {
    const query = `query { me { username, mySatellites { id designation } } }`;
    try {
        const data = await graphqlCall(query);
        document.getElementById('user-name').innerText = data.me.username;
        
        const list = document.getElementById('my-sat-list');
        list.innerHTML = '';
        if (data.me.mySatellites.length === 0) {
            list.innerHTML = '<tr><td colspan="3" style="color: #64748b; text-align:center;">No assets deployed.</td></tr>';
        } else {
            data.me.mySatellites.forEach(sat => {
                list.innerHTML += `<tr>
                    <td>${sat.id}</td>
                    <td>${sat.designation}</td>
                    <td><span class="success-text">Online</span></td>
                </tr>`;
            });
        }
    } catch (e) {
        console.error("Failed to load dashboard:", e);
    }
}

window.viewData = async function(targetId) {
    const output = document.getElementById('telemetry-output');
    output.innerHTML = 'Establishing secure link...';
    
    const query = `
        query GetSatData($id: ID!) {
            satelliteInfo(id: $id) {
                telemetryData { sensorType rawData }
            }
        }
    `;

    try {
        const data = await graphqlCall(query, { id: targetId });
        if (data.satelliteInfo && data.satelliteInfo.telemetryData) {
            let text = `--- DOWNLINK ESTABLISHED ---\n`;
            data.satelliteInfo.telemetryData.forEach(t => {
                text += `[${t.sensorType}]: ${t.rawData}\n`;
            });
            output.innerHTML = `<span class="success-text">${text}</span>`;
        }
    } catch (e) {
        output.innerHTML = `<span class="error-text">CONNECTION TERMINATED\n${e.message}</span>`;
    }
}

document.getElementById('btn-launch').addEventListener('click', async () => {
    const designation = document.getElementById('sat-name').value || "CIVILIAN-PROBE-1";
    const mutation = `mutation { launchCubeSat(designation: "${designation}") { id } }`;
    
    try {
        await graphqlCall(mutation);
        document.getElementById('sat-name').value = '';
        loadDashboard();
    } catch (e) {
        alert("Launch failed: " + e.message);
    }
});

document.getElementById('btn-refresh').addEventListener('click', loadDashboard);
window.addEventListener('DOMContentLoaded', loadDashboard);
'''

with open("project_helios/frontend/app.js", "w") as f:
    f.write(app_js)

print("app.js written")

app.js written


In [ ]:
import subprocess
import shutil

node_path = shutil.which("node")
npm_path = shutil.which("npm")

if node_path and npm_path:
    print(f"✓ Node.js found: {node_path}")
    print(f"✓ npm found: {npm_path}")
    print("\nProceeding with npm install...")
    result = subprocess.run(
        ["npm", "install"],
        cwd="project_helios/backend",
        capture_output=True,
        text=True
    )
    print(result.stdout[-1500:] if result.stdout else "")
    if result.returncode != 0:
        print("ERRORS:", result.stderr[-1000:])
else:
    print("❌ Node.js not installed.")

✓ Node.js found: /usr/bin/node
✓ npm found: /usr/bin/npm

Proceeding with npm install...

up to date, audited 165 packages in 5s

29 packages are looking for funding
  run `npm fund` for details

7 vulnerabilities (2 moderate, 5 high)

To address issues that do not require attention, run:
  npm audit fix

Some issues need review, and may require choosing
a different dependency.

Run `npm audit` for details.



In [ ]:
import subprocess
import time
import os

# Kill any existing server on port 4000 to prevent EADDRINUSE errors
os.system("fuser -k 4000/tcp 2>/dev/null")

server_process = subprocess.Popen(
    ["node", "server.js"],
    cwd="project_helios/backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(2)
print("Server starting...")
print("Visit: http://localhost:4000/")
print(f"\nProcess PID: {server_process.pid}")

 22547Server starting...
Visit: http://localhost:4000/

Process PID: 24932
